# 01. Data Loading and Inspection

This notebook demonstrates how to load raw data from OpenMX output files, create `Snapshot` objects, and inspect their contents.

In [1]:
import torch
from pathlib import Path
import matplotlib.pyplot as plt

from data.snapshot import Snapshot

## Loading a Snapshot

We can load a `Snapshot` from an OpenMX `.scfout` and `.info.out` file pair using the `Snapshot.from_openmx` class method.

In [4]:
# Paths to the data files
# script_dir = Path(__file__).parent.resolve()

# Load the snapshot
snapshot = Snapshot.from_openmx(
    "../../data/big/silicon/900K/Si_DM",
    "../../data/big/silicon/900K/info.txt"
)

print(snapshot)

Snapshot(
  atoms   = 216 atoms
  keys    = ['Si-Si'] …
  basis   = e3nn
)


The `Snapshot` object contains the Hamiltonian, Overlap, and Density matrices as `BlockMatrix` objects.

In [6]:
snapshot.density.orbital_cfg

OrbitalIrrepConfig(
  Si: 2x0e+2x1o+1x2e
)

In [10]:
hamiltonian = snapshot.hamiltonian
overlap = snapshot.overlap
density = snapshot.density

print("Hamiltonian type:", type(hamiltonian))
print("Overlap keys:", overlap.keys())
print("Density atom counts:", density.atom_counts)

Hamiltonian type: <class 'data.block_matrix.BlockMatrix'>
Overlap keys: dict_keys(['Si-Si'])
Density atom counts: Counter({'Si': 216})


In [11]:
density

BlockMatrix(atoms=('Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si', 'Si

In [15]:
snap_water = Snapshot.from_openmx(
    "../../data/small/H2O/original/H2O.matrix",
    "../../data/small/H2O/original/H2O.info.out"
)
print(snap_water)

Snapshot(
  atoms   = HHHHOO
  keys    = ['H-H', 'H-O', 'O-H', 'O-O']
  basis   = e3nn
)


In [16]:
snap_water.density.atoms

('H', 'H', 'H', 'H', 'O', 'O')

In [21]:
snap_water.density["O-O"].shape

torch.Size([4, 22, 22])

In [28]:
snap_water.density.rotate(
    torch.tensor([
        [0, 1, 0],
        [1, 0, 0],
        [0, 0, 1]
    ])
)

BlockMatrix(atoms=('H', 'H', 'H', 'H', 'O', 'O'), atom_counts=Counter({'H': 4, 'O': 2}), pair_blocks={'H-H': tensor([[[-2.8390e+00, -7.1916e-02,  2.4088e-02,  ..., -1.0617e-01,
          -2.1152e-02,  7.4185e-02],
         [-7.1563e-02,  1.7671e-01,  9.3929e-03,  ..., -5.9544e-02,
          -8.0901e-02, -5.0769e-02],
         [ 2.4096e-02,  9.3944e-03,  2.4117e-04,  ..., -1.9629e-03,
          -4.7375e-03, -1.5268e-03],
         ...,
         [-1.0627e-01, -5.9555e-02, -1.9624e-03,  ...,  1.3944e-02,
           2.6234e-02,  1.8908e-02],
         [-2.1331e-02, -8.0923e-02, -4.7383e-03,  ...,  2.6239e-02,
           3.5495e-02,  2.5295e-02],
         [-7.4119e-02,  5.0756e-02,  1.5247e-03,  ..., -1.8902e-02,
          -2.5287e-02, -1.5088e-02]],

        [[-2.8390e+00, -7.1563e-02,  2.4096e-02,  ..., -1.0627e-01,
          -2.1331e-02, -7.4119e-02],
         [-7.1916e-02,  1.7671e-01,  9.3944e-03,  ..., -5.9555e-02,
          -8.0923e-02,  5.0756e-02],
         [ 2.4088e-02,  9.3929e-03,

## Inspecting a BlockMatrix

We can access individual blocks of a `BlockMatrix` in two ways:
1.  Using global atom indices `(i, j)`.
2.  Using a pair-key string `"A-B"`.

In [ ]:
# Accessing a block using global indices (e.g., from atom 0 to atom 4)
block_0_4 = density[(0, 4)]
print("Shape of density block (0, 4):", block_0_4.shape)

# Accessing all blocks for a pair-key (e.g., "Si-Si")
blocks_Si_Si = density["Si-Si"]
print("Shape of all 'Si-Si' density blocks:", blocks_Si_Si.shape)

### Diagonal and Off-Diagonal Blocks

The `diag()` and `offdiag()` methods provide easy access to the diagonal and off-diagonal blocks of a `BlockMatrix`.

In [ ]:
diag_blocks = density.diag()
offdiag_blocks = density.offdiag()

print("Diagonal keys:", diag_blocks.keys())
print("Off-diagonal keys:", offdiag_blocks.keys())

### Dense Representation

For visualization and debugging, we can convert a `BlockMatrix` to a dense `torch.Tensor`.

In [ ]:
dense_density = density.to_dense()

plt.figure(figsize=(6, 6))
plt.imshow(dense_density, cmap="viridis")
plt.title("Dense Density Matrix")
plt.colorbar()
plt.show()

## Physics Observables

The `Snapshot` class provides methods to compute physical observables like the total energy and the number of electrons.

In [29]:
energy = snapshot.get_energy()
num_electrons = snapshot.get_number_of_electrons()

print(f"Total energy: {energy.item():.4f} eV")
print(f"Number of electrons: {num_electrons.item():.4f}")

Total energy: -262.2087 eV
Number of electrons: 863.9997


## Filtering by Distance

We can filter the snapshot to include only edges within a certain cutoff radius. This is useful for creating graphs with different levels of connectivity.

In [30]:
print("Number of electrons before filtering:", snapshot.get_number_of_electrons().item())

# Filter the snapshot to include only edges up to 3.0 Å
filtered_snapshot = snapshot.filter_by_distance(5.0)

print("Number of electrons after filtering:", filtered_snapshot.get_number_of_electrons().item())

Number of electrons before filtering: 863.9996948242188
Number of electrons after filtering: 859.9149780273438
